# Phase 3: Fine-Tune DistilBERT Classifier

**Project:** AI Mental Health Support Chatbot  
**Goal:** Fine-tune `distilbert-base-uncased` on the Phase 2 train/validation/test splits and compare against the LinearSVC baseline (Validation Macro F1 = 0.7392, Test Macro F1 = 0.7165).

> **Run this notebook on Google Colab with GPU.** Full training is blocked on CPU-only machines by design.

## 1. Environment Setup (Colab)
Mount Google Drive, verify GPU availability, clone the repository, and install dependencies.

In [ ]:
import os
import sys
import json
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

print(f"Running in Colab: {IN_COLAB}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# --- Google Drive paths (edit if needed) ---
DRIVE_ROOT = "/content/drive/MyDrive/mental-health-chatbot"
SPLITS_DIR = os.path.join(DRIVE_ROOT, "data/processed/splits")
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "models/distilbert")

REPO_URL = "https://github.com/ArihantJain007/Mental_health-bot.git"
PROJECT_DIR = "/content/Mental_health-bot"

os.makedirs(SPLITS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Drive splits dir:", SPLITS_DIR)
print("Drive output dir:", OUTPUT_DIR)

In [ ]:
if not os.path.isdir(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt

## 2. Verify Phase 2 Splits on Google Drive
Upload the **exact** Phase 2 split files to Drive if they are not already present:
- `train.csv` (40,844 rows)
- `validation.csv` (5,105 rows)
- `test.csv` (5,106 rows)

Do **not** regenerate splits in Colab — use the persisted Phase 2 files.

In [ ]:
import pandas as pd

required_files = ["train.csv", "validation.csv", "test.csv"]
expected_rows = {"train.csv": 40844, "validation.csv": 5105, "test.csv": 5106}

for fname in required_files:
    fpath = os.path.join(SPLITS_DIR, fname)
    if not os.path.exists(fpath):
        raise FileNotFoundError(
            f"Missing {fpath}. Upload Phase 2 split files to Google Drive before training."
        )
    row_count = len(pd.read_csv(fpath))
    assert row_count == expected_rows[fname], f"{fname} row count mismatch: {row_count} != {expected_rows[fname]}"
    print(f"{fname}: {row_count} rows OK")

print("\nAll Phase 2 splits verified on Google Drive.")

## 3. Optional Local Smoke Test (CPU-safe)
This lightweight check validates tokenizer, model init, class weights, forward pass, weighted loss, checkpoint save/load, and sample inference. It does **not** run full training.

In [ ]:
!python -m src.models.distilbert_classifier \
    --smoke-test \
    --data-dir "{SPLITS_DIR}" \
    --output-dir "{OUTPUT_DIR}"

## 4. Full DistilBERT Fine-Tuning (CUDA GPU Required)
Trains for 3 epochs with:
- `max_length=256`
- class-weighted `CrossEntropyLoss` (train labels only)
- validation **Macro F1** checkpoint selection
- checkpoint save/resume support
- **single** held-out test evaluation after model selection

Artifacts are written to Google Drive under `models/distilbert/`:
- `checkpoints/`
- `best_model/`
- `tokenizer/`
- `label_mapping.json`
- `metadata.json`
- `plots/`

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for full training. In Colab: Runtime -> Change runtime type -> GPU"
    )

!python -m src.models.distilbert_classifier \
    --full-train \
    --data-dir "{SPLITS_DIR}" \
    --output-dir "{OUTPUT_DIR}" \
    --num-epochs 3 \
    --batch-size 16 \
    --lr 2e-5

## 5. Resume Training from Latest Checkpoint (Optional)
Use this cell if training was interrupted. It resumes from the newest checkpoint in `checkpoints/`.

In [ ]:
!python -m src.models.distilbert_classifier \
    --full-train \
    --resume \
    --data-dir "{SPLITS_DIR}" \
    --output-dir "{OUTPUT_DIR}" \
    --num-epochs 3 \
    --batch-size 16 \
    --lr 2e-5

## 6. Review Saved Metrics & Sample Inference
After full training completes, inspect `metadata.json` and run sample inference with the best saved model.

In [ ]:
from transformers import AutoModelForSequenceClassification, DistilBertTokenizerFast
from src.models.distilbert_classifier import predict_statement, get_canonical_label_mapping

metadata_path = os.path.join(OUTPUT_DIR, "metadata.json")
best_model_dir = os.path.join(OUTPUT_DIR, "best_model")
tokenizer_dir = os.path.join(OUTPUT_DIR, "tokenizer")

if os.path.exists(metadata_path):
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print(json.dumps(metadata, indent=2))
else:
    print("metadata.json not found yet — run full training first.")

if os.path.isdir(best_model_dir):
    label2id, id2label = get_canonical_label_mapping()
    model = AutoModelForSequenceClassification.from_pretrained(best_model_dir)
    tokenizer = DistilBertTokenizerFast.from_pretrained(tokenizer_dir)

    sample = "I feel deeply anxious and overwhelmed by everything today."
    result = predict_statement(sample, model, tokenizer, label2id, id2label)
    print("\nSample inference:")
    print(result)

## 7. Compare Against Phase 2 LinearSVC Baseline
DistilBERT should be compared using validation Macro F1 for model selection and test Macro F1 for final reporting.

In [ ]:
BASELINE_VAL_MACRO_F1 = 0.7392
BASELINE_TEST_MACRO_F1 = 0.7165

if os.path.exists(metadata_path):
    val_f1 = metadata["validation_best_metrics"]["macro_f1"]
    test_f1 = metadata.get("test_metrics", {}).get("macro_f1")

    print(f"LinearSVC Validation Macro F1: {BASELINE_VAL_MACRO_F1:.4f}")
    print(f"DistilBERT Validation Macro F1: {val_f1:.4f}")
    print(f"Delta (Val): {val_f1 - BASELINE_VAL_MACRO_F1:+.4f}")

    if test_f1 is not None:
        print(f"\nLinearSVC Test Macro F1: {BASELINE_TEST_MACRO_F1:.4f}")
        print(f"DistilBERT Test Macro F1: {test_f1:.4f}")
        print(f"Delta (Test): {test_f1 - BASELINE_TEST_MACRO_F1:+.4f}")
    else:
        print("\nTest metrics not available yet.")
else:
    print("Run full training to populate comparison metrics.")